In [ ]:
# 1. Imports
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import data_loader, features, econometrics, config

# 2. Ingest and Engineer Data
target_etf = config.TARGET_ETFS[0]
raw_data_dict = data_loader.fetch_raw_data()
raw_df = raw_data_dict[target_etf]

# Engineer our stationary features and AR(1) Macro Surprises
stat_df = features.engineer_stationary_features(raw_df, etf_name=target_etf)

# 3. Define the VAR System
# We will use Squared Log Returns as our proxy for Sector Volatility
target_volatility = 'Sq_Log_Return'

# Let's test two specific macro shocks against the Tech Sector
# Ensure these match the exact column names generated by your dynamic features.py
macro_shocks_to_test = ['CPI_Surprise', 'FedFunds_Diff']

# Clean the data for VAR (Order is critical here!)
var_data = econometrics.prepare_var_data(stat_df, target_vol_col=target_volatility, macro_cols=macro_shocks_to_test)

# 4. Granger Causality Tests
# This tests the statistical significance: Does past CPI actually predict future Volatility?
print("="*60)
print(f"GRANGER CAUSALITY TESTS: Macro Shocks -> {target_etf} Volatility")
print("(* = 10% sig, ** = 5% sig, *** = 1% sig)")
print("="*60)
econometrics.test_granger_causality(var_data, max_lag=5)

In [ ]:
# 5. Structural VAR and Impulse Response Functions
# This creates the visual simulation of the shock decaying over time
print("\n" + "="*60)
print(f"IMPULSE RESPONSE FUNCTIONS (Orthogonalized Shock Simulation)")
print("="*60)
var_results, irf = econometrics.fit_var_and_plot_irf(var_data, max_lags=5, irf_periods=15)

In [ ]:
# 6. Historical Alignment (Did the model catch the crises?)
print("\n" + "="*60)
print(f"HISTORICAL SHOCK ALIGNMENT")
print("="*60)
econometrics.plot_historical_fit(var_results, var_data)